In [1]:
!pip install -q -U weaviate-client pypdf sentence-transformers groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 656.2/656.2 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.6/739.6 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 1.7 MB/s eta 0:00:00


In [65]:
# import statements
import os
from pypdf import PdfReader

import weaviate
from weaviate.classes.init import Auth
from weaviate.classes.config import Configure, Property, DataType
from weaviate.classes.query import MetadataQuery, Filter
from weaviate.classes.data import DataObject

from sentence_transformers import SentenceTransformer, CrossEncoder

from google.colab import userdata
from groq import Groq

In [66]:
WEAVIATE_URL=userdata.get("WEAVIATE_URL")
WEAVIATE_API_KEY=userdata.get("WEAVIATE_API_KEY")
GROQ_API_KEY = userdata.get("GROQ_API_KEY")

In [67]:
# weaviate connection establishment

client = weaviate.connect_to_weaviate_cloud(
    cluster_url = WEAVIATE_URL,
    auth_credentials = Auth.api_key(WEAVIATE_API_KEY)
)
print("weaviate Connection",client.is_ready())

weaviate Connection True


In [37]:
# collection - reset, creation

collection_name = 'research_paper'

if client.collections.exists(collection_name):
  print("collection already exists! - so resetting it")
  client.collections.delete(collection_name)

research_paper_collection = client.collections.create(
    name = collection_name,
    vector_config = Configure.Vectors.self_provided(),
    properties = [
        Property(name="doc_name", data_type=DataType.TEXT),
        Property(name="content", data_type=DataType.TEXT),
        Property(name="category", data_type=DataType.TEXT),
        Property(name="page", data_type=DataType.INT),
        Property(name="chunk_id", data_type=DataType.INT)
    ]
)

collection already exists! - so resetting it


In [27]:
source_path = '/content/knowledge_source'
print(os.listdir(source_path))

['NIPS-2017-attention-is-all-you-need-Paper.pdf', '1301.3781v3.pdf']


In [31]:
embedder_model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [38]:
# Data Ingestion
# Chunking ; Overlapping
# Text Embedding

def chunk_text(text, chunk_size=500, overlap=50):
  chunks = []
  words = text.split(" ")
  for i in range(0, len(words), chunk_size - overlap):
    chunk = " ".join(words[i: i + chunk_size])
    chunks.append(chunk)
  return chunks

def ingest_data (folder_path):
  batch = []

  for file in os.listdir(folder_path):
    if file.endswith('.pdf'):
      print(f'Processing : {file}')
      reader = PdfReader(os.path.join(folder_path, file))
      for page_num, page_text in enumerate(reader.pages,1):
        page_text = page_text.extract_text().strip() or ""

        if not page_text:
          continue

        print(f'Processing : {file} - Page {page_num}')
        chunks = chunk_text(page_text)

        print(f'Processing : {file} - Creating Embedding {page_num}')
        for chunk_id, chunk in enumerate(chunks, 1):
          vector = embedder_model.encode(chunk).tolist()
          batch.append(
              DataObject(
                  properties={
                      "doc_name": file.replace(".pdf", ""),
                      "content": chunk,
                      "category": "Transformer" if "attention" in file else "Embedding",
                      "page": page_num,
                      "chunk_id": chunk_id,
                  },
                  vector=vector
              )
          )
    else:
      pass
    print(f'Processing : {file} - Completed')

  print(f'Batches count : {len(batch)} : Ingesting')
  research_paper_collection.data.insert_many(batch)
  print(f'Batches count : {len(batch)} : Completed')

ingest_data(source_path)

Processing : NIPS-2017-attention-is-all-you-need-Paper.pdf
Processing : NIPS-2017-attention-is-all-you-need-Paper.pdf - Page 1
Processing : NIPS-2017-attention-is-all-you-need-Paper.pdf - Creating Embedding 1
Processing : NIPS-2017-attention-is-all-you-need-Paper.pdf - Page 2
Processing : NIPS-2017-attention-is-all-you-need-Paper.pdf - Creating Embedding 2
Processing : NIPS-2017-attention-is-all-you-need-Paper.pdf - Page 3
Processing : NIPS-2017-attention-is-all-you-need-Paper.pdf - Creating Embedding 3
Processing : NIPS-2017-attention-is-all-you-need-Paper.pdf - Page 4
Processing : NIPS-2017-attention-is-all-you-need-Paper.pdf - Creating Embedding 4
Processing : NIPS-2017-attention-is-all-you-need-Paper.pdf - Page 5
Processing : NIPS-2017-attention-is-all-you-need-Paper.pdf - Creating Embedding 5
Processing : NIPS-2017-attention-is-all-you-need-Paper.pdf - Page 6
Processing : NIPS-2017-attention-is-all-you-need-Paper.pdf - Creating Embedding 6
Processing : NIPS-2017-attention-is-all-y

In [60]:
# RETRIVAL

def retrieve_docs (query, category, alpha = 0.5, limit = 50):
  query = query.strip()
  category = category.strip()
  if not query:
    return []
  else:
    query_vector = embedder_model.encode(query).tolist()
    apply_filter = bool(category)
    response = research_paper_collection.query.hybrid(
        query = query,
        vector = query_vector,
        limit = limit,
        alpha = alpha,
        filters = Filter.by_property('category').equal(category) if apply_filter else None,
        return_metadata = MetadataQuery(score=True, distance=True)
    )
    print('Retrieved information for answering ', len(response.objects))
    return response.objects


In [59]:
# RERANKING (OPTIONAL)
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def reranking_retrieved (query, retrieved_docs, top_n=5):
  query = query.strip()
  docs = [doc.properties["content"] for doc in retrieved_docs]
  pairs = [[query, doc] for doc in docs]

  scores = cross_encoder.predict(pairs)
  ranked = sorted(list(zip(retrieved_docs, scores)), key = lambda x : x[1], reverse=True)[:top_n]

  print('Finalized Retrieved information for answering ', len(ranked))
  return ranked

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [52]:
# AUGMENTATION

def build_prompt(query, retrieved_docs):
  context_parts = []
  for doc, score in retrieved_docs:
    context_parts.append(
      f"""
      Source: {doc.properties["doc_name"]}
      Page: {doc.properties["page"]}
      Chunk: {doc.properties["chunk_id"]}
      Score: {score}

      {doc.properties["content"]}
      """
    )

  context = "\n\n---\n\n".join(context_parts)
  prompt = f"""
    You are a helpful Research Paper AI assistant.

    Answer the question using ONLY the provided context.

    If the answer cannot be found in the context,
    say that you don't have enough information.

    Context:
    {context}

    Question:
    {query}

    Answer:
  """
  return prompt

In [63]:
# GENERATION

def generate_response (prompt):
  groq_client = Groq(api_key=GROQ_API_KEY)

  response = groq_client.chat.completions.create(
      model="openai/gpt-oss-20b",
      messages=[
          {
              "role": "user",
              "content": prompt
          }
      ]
  )
  print(f'No of responses {len(response.choices)} generated')
  return response.choices[0].message.content

In [55]:
# WIRING
def ask_user():
  user_query = input("Enter your query: ").strip()
  category = input("Know the paper (Transformer/Embedding)? ").strip()
  if not user_query:
    print("Invalid input - retry")
  else:
    retrieved_content = retrieve_docs(user_query, category, alpha=0.8, limit=10)
    rerank_content = reranking_retrieved(user_query, retrieved_content, 5)
    prompt = build_prompt(user_query, rerank_content)
    response = generate_response(prompt)
    print("\nTop 1 Response : \n")
    print(response)

In [61]:
ask_user()

Enter your query: what is embedding in simple terms with examples
Know the paper (Transformer/Embedding)? Embedding
Retrieved information for answering  10
Finalized Retrieved information for answering  5
No of responses 1 generated

Top 1 Response : 

**Embedding (in the context of this paper)**  
An embedding is a **continuous vector representation of a word**.  
Each word is turned into a point in a high‑dimensional space (the paper uses 300‑dimensional vectors).  
Words that are similar in meaning or usage end up close to each other in that space, while dissimilar words are far apart.

**How it works in the paper**  
The paper learns these vectors with two simple neural‑network models:

| Model | What it predicts | Input |
|-------|------------------|-------|
| CBOW | Current word from surrounding words | Context words |
| Skip‑gram | Surrounding words from the current word | Current word |

These models produce the word vectors (embeddings) shown in Table 4 of the paper.

**Exampl